# pygeoinf3D — Demo 4: Function Providers

This notebook surveys the **provider** classes in `pygeoinf3D`.  A *provider* is
an object that generates a parametric family of `Function3D` objects on demand,
exposing a common interface used by `SOLAOperator3D`, `FunctionToKernelAdapter`,
and other parts of the pipeline.

**Providers covered:**

| Class | Description |
|-------|-------------|
| `TensorProductProvider` | Laplacian eigenfunctions on a box (spectral basis) |
| `GaussianProvider3D` | Gaussian bumps centred on a grid of points |
| `BumpProvider3D` | Smooth compactly-supported bump functions |
| `BoxcarProvider3D` | Indicator-like boxcar functions |
| `FunctionToKernelAdapter` | Wraps any `FunctionProvider3D` as a `KernelProvider3D` |

**Next demo:** `5_gradient_operator_3d_demo.ipynb`  ·  `6_laplacian_3d_demo.ipynb`

## 1. Imports and Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pygeoinf3d import BoxDomain, Lebesgue3D, TensorProductProvider, FunctionToKernelAdapter
from pygeoinf3d.plot import plot_function3d_axis_slice

# --- optional providers ---
try:
    from pygeoinf3d.providers.function_providers import GaussianProvider3D
    GAUSSIAN_AVAILABLE = True
except ImportError:
    GaussianProvider3D = None
    GAUSSIAN_AVAILABLE = False

try:
    from pygeoinf3d.providers.function_providers import BumpProvider3D
    BUMP_AVAILABLE = True
except ImportError:
    BumpProvider3D = None
    BUMP_AVAILABLE = False

try:
    from pygeoinf3d.providers.function_providers import BoxcarProvider3D
    BOXCAR_AVAILABLE = True
except ImportError:
    BoxcarProvider3D = None
    BOXCAR_AVAILABLE = False

print(f"GaussianProvider3D available : {GAUSSIAN_AVAILABLE}")
print(f"BumpProvider3D     available : {BUMP_AVAILABLE}")
print(f"BoxcarProvider3D   available : {BOXCAR_AVAILABLE}")
print(f"FunctionToKernelAdapter      : always available")

# Shared domain and L² space
box   = BoxDomain(np.zeros(3), np.ones(3))
space = Lebesgue3D(box)
print(f"\nBoxDomain : {box.bounding_box().low} to {box.bounding_box().high}")
print(f"Lebesgue3D dim (basis-free) : {space.dim}")


## 2. TensorProductProvider — Overview and Eigenvalue Spectrum

`TensorProductProvider` builds an orthonormal spectral basis from the tensor
product of 1-D Laplacian eigenfunctions on the box.  The eigenfunctions satisfy
either **Dirichlet** ($\phi = 0$ at boundaries) or **Neumann**
($\partial_n \phi = 0$ at boundaries) boundary conditions.

For a unit cube $[0,1]^3$ with **Dirichlet** BCs, multi-index $(k_x, k_y, k_z)$
with $k_j \geq 1$ gives:

$$\phi_{k_x k_y k_z}(x,y,z) = 2^{3/2}\,\sin(k_x\pi x)\,\sin(k_y\pi y)\,\sin(k_z\pi z)$$

with eigenvalue $\lambda = \pi^2(k_x^2 + k_y^2 + k_z^2)$.

`truncation=3` means $k_j \in \{1,2,3\}$ for each axis → $3^3 = 27$ modes.

In [ ]:
# ── Dirichlet provider ──────────────────────────────────────────────────────
dirichlet = TensorProductProvider(space, truncation=3, bc_type='dirichlet')

print("=== Dirichlet (truncation=3) ===")
print(f"  dim       : {dirichlet.dim}  (expected 27 = 3³)")
print(f"  truncation: {dirichlet.truncation}")
print(f"  bc_type   : {dirichlet.bc_type}")

# ── Neumann provider ─────────────────────────────────────────────────────────
neumann = TensorProductProvider(space, truncation=3, bc_type='neumann')

print("\n=== Neumann (truncation=3) ===")
print(f"  dim       : {neumann.dim}  (includes k=0 constant mode)")

# ── Side-by-side eigenvalue table ────────────────────────────────────────────
print("\n{:>3}  {:>14}  {:>10}  {:>14}  {:>10}".format(
    'i', 'Dirichlet mi', 'lam_D', 'Neumann mi', 'lam_N'))
print("-" * 60)
for i in range(9):
    mi_d  = dirichlet.get_multi_index(i)
    lam_d = dirichlet.get_eigenvalue(i)
    mi_n  = neumann.get_multi_index(i)
    lam_n = neumann.get_eigenvalue(i)
    print(f"{i:3d}  {str(mi_d):>14}  {lam_d:10.4f}  {str(mi_n):>14}  {lam_n:10.4f}")


**Key observations:**
- The Dirichlet ground state has $\lambda_0 \approx 29.6 = 3\pi^2$ (all three
  indices equal to 1); the zero eigenvalue is excluded by the boundary condition.
- Neumann BCs admit $k_j = 0$ (constant functions), giving $\lambda_0 = 0$ — the
  constant mode — and a much denser low-frequency spectrum.
- Both spectra show three-fold degeneracy for modes like $(1,1,2)$ due to the
  cubic symmetry of the domain.

## 3. TensorProductProvider — Eigenfunction Visualisation

We use `truncation=2` to get exactly $2^3 = 8$ eigenfunctions and display each
as a coloured heatmap on the $z = 0.5$ midplane.

In [ ]:
prov2 = TensorProductProvider(space, truncation=2, bc_type='dirichlet')
print(f"truncation=2  →  dim = {prov2.dim}  (2³ = 8 eigenfunctions)")

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
axes = axes.ravel()

for i in range(8):
    phi   = prov2.get_eigenfunction(i)
    mi    = prov2.get_multi_index(i)
    lam_i = prov2.get_eigenvalue(i)
    _, _, payload = plot_function3d_axis_slice(
        phi, axis='z', coord=0.5, resolution=60,
        show_plot=False, ax=axes[i],
    )
    axes[i].set_title(f"φ_{i}  {mi}\nλ={lam_i:.1f}", fontsize=8)
    axes[i].set_xlabel('x', fontsize=7)
    axes[i].set_ylabel('y', fontsize=7)
    axes[i].tick_params(labelsize=6)

fig.suptitle("Dirichlet eigenfunctions (truncation=2) at z = 0.5", fontsize=11)
fig.tight_layout()
plt.show()


In [ ]:
# ── Boundary-vanishing check ─────────────────────────────────────────────────
n_check = 5
y_vals  = np.linspace(0.1, 0.9, n_check)
z_vals  = np.linspace(0.1, 0.9, n_check)
yy, zz  = np.meshgrid(y_vals, z_vals)
pts_near_x0 = np.column_stack([
    np.full(n_check**2, 1e-6),
    yy.ravel(),
    zz.ravel(),
])

print("Dirichlet boundary check — values near x = 0 boundary")
print(f"{'i':>3}  {'multi-index':>14}  {'max |phi_i| near x=0':>22}")
print("-" * 45)
for i in range(8):
    phi  = prov2.get_eigenfunction(i)
    vals = phi(pts_near_x0)
    mi   = prov2.get_multi_index(i)
    print(f"{i:3d}  {str(mi):>14}  {np.max(np.abs(vals)):22.2e}")


## 4. TensorProductProvider — Eigenvalue Ordering and Tensor Structure

With `truncation=4` we have $4^3 = 64$ modes.  The provider sorts them by
ascending eigenvalue (with lexicographic multi-index tie-breaking), revealing the
characteristic staircase spectrum of a 3-D Laplacian.

In [ ]:
prov4 = TensorProductProvider(space, truncation=4, bc_type='dirichlet')
print(f"truncation=4  →  dim = {prov4.dim}  (4³ = 64 eigenfunctions)")

eigenvalues = np.array([prov4.get_eigenvalue(i) for i in range(prov4.dim)])

fig, ax = plt.subplots(figsize=(10, 3))
ax.scatter(np.arange(prov4.dim), eigenvalues, s=20, c='steelblue', alpha=0.85, zorder=3)
ax.set_xlabel("Mode index $i$ (sorted by $\\lambda$)")
ax.set_ylabel("Eigenvalue $\\lambda_i$")
ax.set_title("Dirichlet spectrum of $-\\nabla^2$ (truncation=4, unit cube)")
ax.grid(True, linestyle='--', alpha=0.5)
fig.tight_layout()
plt.show()

print("\nFirst 16 modes (sorted by eigenvalue):")
print(f"{'i':>3}  {'multi-index (kx,ky,kz)':>24}  {'lam':>10}  {'kx2+ky2+kz2':>12}")
print("-" * 60)
for i in range(16):
    mi  = prov4.get_multi_index(i)
    lam = prov4.get_eigenvalue(i)
    ss  = mi[0]**2 + mi[1]**2 + mi[2]**2
    print(f"{i:3d}  {str(mi):>24}  {lam:10.4f}  {ss:12d}")


**Degenerate eigenvalues** arise whenever permutations of $(k_x, k_y, k_z)$
give the same sum of squares.  For a unit cube this happens at the *very first
excited level*: modes $(1,1,2)$, $(1,2,1)$, $(2,1,1)$ all have
$k_x^2+k_y^2+k_z^2 = 6$, so $\lambda = 6\pi^2 \approx 59.2$.

The staircase shape of the spectrum reflects the fact that mode count grows
as $\sim \lambda^{3/2}$ (Weyl's law in 3-D).

## 5. GaussianProvider3D

`GaussianProvider3D` builds a function family from normalised Gaussian bumps
centred at specified points:

$$\phi_i(\mathbf{x}) \propto \exp\!\left(-\frac{\|\mathbf{x} - \mathbf{c}_i\|^2}{2\sigma_i^2}\right)$$

Each function in the family corresponds to one centre, making this provider
ideal for soft, overlapping kernel functions used in SOLA-type inversions.

In [ ]:
if GAUSSIAN_AVAILABLE:
    centers_g = np.array([
        [0.25, 0.25, 0.5],
        [0.75, 0.25, 0.5],
        [0.25, 0.75, 0.5],
        [0.75, 0.75, 0.5],
    ])
    gauss_prov = GaussianProvider3D(box, centers=centers_g, sigmas=0.12)
    print(f"GaussianProvider3D  dim = {gauss_prov.dim}  (one per centre)")

    fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
    for i in range(4):
        phi = gauss_prov.get_function(i)
        _, _, _ = plot_function3d_axis_slice(
            phi, axis='z', coord=0.5, resolution=60,
            show_plot=False, ax=axes[i],
        )
        axes[i].set_title(f"Gaussian phi_{i}\ncenter={centers_g[i]}", fontsize=8)
        axes[i].tick_params(labelsize=6)
    fig.suptitle("GaussianProvider3D — z = 0.5 slice", fontsize=11)
    fig.tight_layout()
    plt.show()
else:
    print("GaussianProvider3D is not available in this installation.")


## 6. BumpProvider3D

`BumpProvider3D` constructs smooth, compactly supported bump functions.  Unlike
Gaussian functions, bump functions are *exactly* zero outside a prescribed
radius, making them useful when strict localisation is required.

The smooth bump kernel is:

$$b(r) = \exp\!\left(-\frac{1}{1 - (r/R)^2}\right) \cdot \mathbf{1}_{r < R}$$

In [ ]:
if BUMP_AVAILABLE:
    xs = [0.25, 0.75]
    ys = [0.25, 0.75]
    centers_b = np.array([[x, y, 0.5] for x in xs for y in ys])
    bump_prov = BumpProvider3D(box, centers=centers_b, widths=0.25)
    print(f"BumpProvider3D  dim = {bump_prov.dim}  (one per centre)")

    fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
    for i in range(4):
        phi = bump_prov.get_function(i)
        _, _, _ = plot_function3d_axis_slice(
            phi, axis='z', coord=0.5, resolution=60,
            show_plot=False, ax=axes[i],
        )
        axes[i].set_title(f"Bump phi_{i}\ncenter={centers_b[i]}", fontsize=8)
        axes[i].tick_params(labelsize=6)
    fig.suptitle("BumpProvider3D — z = 0.5 slice", fontsize=11)
    fig.tight_layout()
    plt.show()

    phi0         = bump_prov.get_function(0)
    probe_far    = np.array([[0.5, 0.5, 0.5]])
    probe_centre = centers_b[[0]]
    print(f"\nphi_0 at (0.5,0.5,0.5)       : {phi0(probe_far)[0]:.4f} (outside bump 0 support)")
    print(f"phi_0 at centre (0.25,0.25,0.5): {phi0(probe_centre)[0]:.4f} (at bump 0 centre)")
else:
    print("BumpProvider3D is not available in this installation.")


## 7. FunctionToKernelAdapter

`FunctionToKernelAdapter` bridges **function providers** and **kernel providers**.
Given any `FunctionProvider3D` and a `Lebesgue3D` space, it wraps each function
$\phi_i$ into a `VolumeKernel3D`, turning the function provider into a
`KernelProvider3D` compatible with `SOLAOperator3D`.

```
FunctionProvider3D → FunctionToKernelAdapter → KernelProvider3D
                                                       ↓
                                              VolumeKernel3D objects
                                              (usable in SOLAOperator3D)
```

In [ ]:
if GAUSSIAN_AVAILABLE:
    kernel_adapter = FunctionToKernelAdapter(gauss_prov, space)

    print("FunctionToKernelAdapter wrapping GaussianProvider3D:")
    print(f"  adapter.dim      = {kernel_adapter.dim}   (same as function provider)")

    k0 = kernel_adapter.get_kernel(0)
    print(f"  get_kernel(0)    = {type(k0).__name__}")
    print(f"  k0.kernel type   = {type(k0.kernel).__name__}")

    phi0_fn  = gauss_prov.get_function(0)
    phi0_ker = k0.kernel
    test_pts = np.random.default_rng(0).uniform(0.1, 0.9, (20, 3))
    diff = np.max(np.abs(phi0_fn(test_pts) - phi0_ker(test_pts)))
    print(f"\n  max |phi_0(x) - kernel(x)| over 20 pts = {diff:.2e}  (should be 0)")

    from pygeoinf3d import Function3D
    f_ones = Function3D(box, evaluate_callable=lambda X: np.ones(len(X)))
    val = k0(f_ones)
    print(f"\n  k0(f=1) = {val:.6f}   (integral of Gaussian over unit cube)")
else:
    print("FunctionToKernelAdapter demo skipped: GaussianProvider3D not available.")


## 8. Provider Interface Summary

All provider classes satisfy the `FunctionProvider3D` protocol:

| Attribute / Method | Description |
|--------------------|-------------|
| `.dim` | `int` — total number of functions in the family |
| `.get_function(i)` | Return the `i`-th `Function3D` (0-indexed) |

`TensorProductProvider` additionally exposes:

| Attribute / Method | Description |
|--------------------|-------------|
| `.truncation` | `tuple[int,int,int]` — per-axis mode counts |
| `.bc_type` | `'dirichlet'` or `'neumann'` |
| `.get_eigenvalue(i)` | $\lambda_i$ — Laplacian eigenvalue |
| `.get_eigenfunction(i)` | Alias of `get_function(i)` |
| `.get_multi_index(i)` | `(kx,ky,kz)` — per-axis mode numbers |
| `.get_index((kx,ky,kz))` | Inverse look-up: multi-index → flat index |
| `__iter__` | Yields `(eigenvalue, Function3D)` pairs |

In [ ]:
available_providers = [
    ('TensorProductProvider (Dirichlet, trunc=3)', dirichlet, 'spectral eigenfunction'),
    ('TensorProductProvider (Neumann,   trunc=3)', neumann,   'spectral eigenfunction'),
]
if GAUSSIAN_AVAILABLE:
    available_providers.append(('GaussianProvider3D (4 centres)', gauss_prov, 'Gaussian bump'))
if BUMP_AVAILABLE:
    available_providers.append(('BumpProvider3D (4 centres)', bump_prov, 'smooth bump'))

print(f"{'Provider':50s}  {'dim':>5}  {'get_function(0)':>18}  {'basis type':20s}")
print("-" * 100)
for name, prov, basis in available_providers:
    f0 = prov.get_function(0)
    print(f"{name:50s}  {prov.dim:5d}  {type(f0).__name__:>18}  {basis:20s}")


## 9. Summary

### Providers at a glance

| Provider | Basis type | dim formula | BCs | Extra API |
|----------|-----------|-------------|-----|-----------|
| `TensorProductProvider` | Laplacian eigenfunctions | $n_x \times n_y \times n_z$ | Dirichlet or Neumann | `get_eigenvalue`, `get_multi_index`, `__iter__` |
| `GaussianProvider3D` | Gaussian bumps $e^{-r^2/2\sigma^2}$ | one per centre | — | — |
| `BumpProvider3D` | Smooth compactly-supported bumps | one per centre | — | — |
| `BoxcarProvider3D` | Step / indicator functions | one per centre | — | — |
| `FunctionToKernelAdapter` | Wraps any `FunctionProvider3D` | inherits `.dim` | — | `.get_kernel(i)` → `VolumeKernel3D` |

### How providers fit into the pipeline

```
BoxDomain  ──►  Lebesgue3D  ──►  TensorProductProvider (or Gaussian/Bump/…)
                                          │
                              FunctionToKernelAdapter
                                          │
                                  SOLAOperator3D  ──►  inversion
```

### Next demo

**→ `6_laplacian_3d_demo.ipynb`** — `BoxLaplacian`, `InverseBoxLaplacian`, and
`GeneralizedBoxLaplacian`; covariance operators and spectral priors.